# Create histone features

In [1]:
# Import Library
import polars as pl
import os
import numpy as np
import torch

## Define Required Functions

### Define Gene Expression schema

In [2]:
# Define the gene expression schema
schema = pl.Schema({
        'chromosome_name': pl.String,
        'start': pl.Int64,
        'end': pl.Int64,
        'gene_id': pl.String,
        'E066': pl.Float64,
        'strand': pl.Int64,
        'label': pl.Int64,
        'external_gene_name': pl.String,
        'start_position': pl.Int64,
        'end_position': pl.Int64,
        'tss': pl.Int64
})

In [3]:
DATASET_PATH = '../dataset/E066/'

### Define the function for getting histone features

In [4]:
def get_histone_features(genes_df, histone_df, histone_name):
    
    # Create a sequence of window starts and window ends for each gene
    genes_with_windows = genes_df.with_columns([
            pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_starts')
        ]).explode('window_starts')

    genes_with_windows = genes_with_windows.with_columns([
            (pl.col('window_starts') + 100).alias('window_ends')
        ])

    # Join genes with histone data
    joined = genes_with_windows.join(
        histone_df,
        left_on='gene_id',
        right_on='gene_id',
        how='left'
    )

    # Filter and calculate average signal value
    result = joined.filter(
        (pl.col('chromStart') <= pl.col('window_starts')) & 
        (pl.col('chromEnd') >= pl.col('window_ends')) |
        (pl.col('chromStart') >= pl.col('window_starts')) & 
        (pl.col('chromStart') <= pl.col('window_ends')) & 
        (pl.col('chromEnd') >= pl.col('window_ends')) |
        (pl.col('chromStart') <= pl.col('window_starts')) & 
        (pl.col('chromEnd') >= pl.col('window_starts')) & 
        (pl.col('chromEnd') <= pl.col('window_ends'))
    ).group_by(['gene_id', 'window_starts'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_starts'])

    # Find the gene without histone match
    gene_wo_histone = genes_with_windows.filter(~pl.col('window_starts').is_in(result['window_starts']))

    # Restructured the dataframe so it can be merged with results
    gene_wo_histone = gene_wo_histone.with_columns(
        chromosome_name_right = pl.lit(None).cast(pl.String),
        start_right = pl.lit(0).cast(pl.Int64),
        end_right =  pl.lit(0).cast(pl.Int64),
        E066_right = pl.lit(0.0).cast(pl.Float64),
        strand_right = pl.lit(0).cast(pl.Int64),
        label_right = pl.lit(0).cast(pl.Int64),
        external_gene_name_right = pl.lit(None).cast(pl.String),
        start_position_right = pl.lit(0).cast(pl.Int64),
        end_position_right = pl.lit(0).cast(pl.Int64),
        tss_right = pl.lit(0).cast(pl.Int64),
        chrom = pl.lit(None).cast(pl.String),
        chromStart = pl.lit(0).cast(pl.Int64),
        chromEnd = pl.lit(0).cast(pl.Int64),
        name = pl.lit(None).cast(pl.String),
        score = pl.lit(0).cast(pl.Int64),
        strand_peak = pl.lit(None).cast(pl.String),
        signalValue = pl.lit(0.0).cast(pl.Float64),
        pValue = pl.lit(0.0).cast(pl.Float64),
        qValue = pl.lit(0.0).cast(pl.Float64),
        peak = pl.lit(0).cast(pl.Int64),
        startBucket = pl.lit(0.0).cast(pl.Float64),
        endBucket = pl.lit(0.0).cast(pl.Float64)
    )

    # Grouping so it can be merged with result dataframe
    gene_wo_histone = gene_wo_histone.group_by(['gene_id', 'window_starts'], maintain_order=True).agg([
        pl.col('signalValue').mean().alias(histone_name)
    ]).sort(['gene_id', 'window_starts'])

    # Merging with the result dataframe
    result.extend(gene_wo_histone)

    # Sorting by window_start
    result = result.sort("window_starts")

    # Final group by
    result = result.group_by(['gene_id'], maintain_order=True).agg(
        pl.col(histone_name)
    ).sort('gene_id')

    result = result.with_columns(
        pl.col(histone_name)
        .list.eval(pl.element().is_not_null() & (pl.element() > 0))
        .list.sum()
        .alias(f"{histone_name}_wc")
    )

    result = result.with_columns(
        pl.col(histone_name).list.len().alias(f"{histone_name}_len")
    )

    return result

### Function for getting empty histone dataframe

In [5]:
def create_empty_dataframe(histone_name):
    
    schema = pl.Schema({
        'gene_id': pl.String,
        histone_name: pl.List(pl.Float64),
        f'{histone_name}_wc': pl.UInt32,
        f'{histone_name}_len': pl.UInt32
    })

    df = pl.DataFrame(schema=schema)

    return df

### Function for getting histone in chunk

In [6]:
# Getting histone in chunk
def get_histone_features_chunk(genes, histones, histone_name):
    
    gene_w_histone = create_empty_dataframe(histone_name)
    
    for i, chunk in enumerate(genes.iter_slices(n_rows=100)):
        result = get_histone_features(chunk, histones, histone_name)
        gene_w_histone.extend(result)

    return gene_w_histone

## Loading the dataset

In [7]:
# Read gene expression file with chromosome and +/- 5k from TSS
E066_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066.bed"), 
                      separator="\t", 
                      schema=schema,
                      has_header=False,
                      skip_rows=0)

In [9]:
E066_pl

chromosome_name,start,end,gene_id,E066,strand,label,external_gene_name,start_position,end_position,tss
str,i64,i64,str,f64,i64,i64,str,i64,i64,i64
"""chrX""",99889988,99899988,"""ENSG00000000003""",73.205,-1,1,"""TSPAN6""",99883667,99894988,99894988
"""chrX""",99834799,99844799,"""ENSG00000000005""",0.191,1,0,"""TNMD""",99839799,99854882,99839799
"""chr20""",49570092,49580092,"""ENSG00000000419""",52.609,-1,1,"""DPM1""",49551404,49575092,49575092
"""chr1""",169858408,169868408,"""ENSG00000000457""",4.733,-1,1,"""SCYL3""",169818772,169863408,169863408
"""chr1""",169626245,169636245,"""ENSG00000000460""",0.942,1,0,"""C1orf112""",169631245,169823221,169631245
…,…,…,…,…,…,…,…,…,…,…
"""chr15""",102280913,102290913,"""ENSG00000259658""",0.212,-1,0,"""RP11-89K11.1""",102277302,102285913,102285913
"""chr15""",97966182,97976182,"""ENSG00000259664""",0.0,-1,0,"""CTD-2147F2.2""",97913601,97971182,97971182
"""chr16""",33642696,33652696,"""ENSG00000259680""",0.071,-1,0,"""RP11-812E19.9""",33647044,33647696,33647696


In [10]:
# Loading histone file
H3K4me1_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me1_df.csv"))
H3K4me3_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K4me3_df.csv"))
H3K9me3_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K9me3_df.csv"))
H3K27me3_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K27me3_df.csv"))
H3K36me3_pl = pl.read_csv(os.path.join(DATASET_PATH, "E066_H3K36me3_df.csv"))

## Find the gene and histone intersection

In [11]:
gene_w_H3K4me1 = get_histone_features_chunk(E066_pl, H3K4me1_pl, 'H3K4me1')
gene_w_H3K4me3 = get_histone_features_chunk(E066_pl, H3K4me3_pl, 'H3K4me3')
gene_w_H3K9me3 = get_histone_features_chunk(E066_pl, H3K9me3_pl, 'H3K9me3')
gene_w_H3K27me3 = get_histone_features_chunk(E066_pl, H3K27me3_pl, 'H3K27me3')
gene_w_H3K36me3 = get_histone_features_chunk(E066_pl, H3K36me3_pl, 'H3K36me3')

In [12]:
gene_w_H3K4me1

gene_id,H3K4me1,H3K4me1_wc,H3K4me1_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",9,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",52,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


In [13]:
gene_w_H3K4me3

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 4.19024, … 0.0]",19,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",36,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


In [14]:
gene_w_H3K9me3

gene_id,H3K9me3,H3K9me3_wc,H3K9me3_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",2,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",2,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[3.0104, 3.0104, … 0.0]",5,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


In [15]:
gene_w_H3K27me3

gene_id,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",5,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


In [16]:
gene_w_H3K36me3

gene_id,H3K36me3,H3K36me3_wc,H3K36me3_len
str,list[f64],u32,u32
"""ENSG00000000003""","[3.39243, 0.0, … 0.0]",1,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[10.55422, 10.55422, … 0.0]",25,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100
…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100


## Join all histones into single dataframe

In [17]:
# Join all histone into single dataframe
gene_w_histone = gene_w_H3K4me3 \
                    .join(gene_w_H3K4me1, on='gene_id') \
                    .join(gene_w_H3K36me3, on='gene_id') \
                    .join(gene_w_H3K9me3, on='gene_id') \
                    .join(gene_w_H3K27me3, on='gene_id')

In [18]:
gene_w_histone

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K36me3,H3K36me3_wc,H3K36me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32
"""ENSG00000000003""","[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",9,100,"[3.39243, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",52,100,"[10.55422, 10.55422, … 0.0]",25,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.0104, 3.0104, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",5,100
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100


## Join with E066 label

In [19]:
# Load E066 dataset
E066_w_label = pl.read_csv(os.path.join(DATASET_PATH, 'E066_merged.csv'))

In [20]:
E066_w_label

gene_id,E066,label,external_gene_name,chromosome_name,start_position,end_position,strand,tss
str,f64,i64,str,str,i64,i64,i64,i64
"""ENSG00000000003""",73.205,1,"""TSPAN6""","""chrX""",99883667,99894988,-1,99894988
"""ENSG00000000005""",0.191,0,"""TNMD""","""chrX""",99839799,99854882,1,99839799
"""ENSG00000000419""",52.609,1,"""DPM1""","""chr20""",49551404,49575092,-1,49575092
"""ENSG00000000457""",4.733,1,"""SCYL3""","""chr1""",169818772,169863408,-1,169863408
"""ENSG00000000460""",0.942,0,"""C1orf112""","""chr1""",169631245,169823221,1,169631245
…,…,…,…,…,…,…,…,…
"""ENSG00000259658""",0.212,0,"""RP11-89K11.1""","""chr15""",102277302,102285913,-1,102285913
"""ENSG00000259664""",0.0,0,"""CTD-2147F2.2""","""chr15""",97913601,97971182,-1,97971182
"""ENSG00000259680""",0.071,0,"""RP11-812E19.9""","""chr16""",33647044,33647696,-1,33647696


In [21]:
E066_w_label = E066_w_label.select(['gene_id', 'E066'])

In [22]:
E066_w_label

gene_id,E066
str,f64
"""ENSG00000000003""",73.205
"""ENSG00000000005""",0.191
"""ENSG00000000419""",52.609
"""ENSG00000000457""",4.733
"""ENSG00000000460""",0.942
…,…
"""ENSG00000259658""",0.212
"""ENSG00000259664""",0.0
"""ENSG00000259680""",0.071


In [24]:
# Create a quantile label
# Define the quartile boundaries
quantile = [0.25, 0.5, 0.75]

In [25]:
# Create the quartile labels
E066_w_qlabel = E066_w_label.with_columns([
    pl.col('E066')
    .qcut(quantile, labels=['0', '1', '2', '3'])
    .alias('label')
])

In [27]:
E066_w_qlabel = E066_w_qlabel.with_columns(pl.col('label').cast(pl.Int64))

In [28]:
E066_w_qlabel

gene_id,E066,label
str,f64,i64
"""ENSG00000000003""",73.205,3
"""ENSG00000000005""",0.191,1
"""ENSG00000000419""",52.609,3
"""ENSG00000000457""",4.733,2
"""ENSG00000000460""",0.942,1
…,…,…
"""ENSG00000259658""",0.212,1
"""ENSG00000259664""",0.0,0
"""ENSG00000259680""",0.071,0


In [29]:
# Join with gene_w_histone
gene_w_histone = gene_w_histone.join(
    E066_w_qlabel,
    on = 'gene_id'
)

In [30]:
gene_w_histone

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K36me3,H3K36me3_wc,H3K36me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E066,label
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,i64
"""ENSG00000000003""","[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",9,100,"[3.39243, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,73.205,3
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,0.191,1
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",52,100,"[10.55422, 10.55422, … 0.0]",25,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,52.609,3
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,4.733,2
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.0104, 3.0104, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,0.942,1
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212,1
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",5,100,0.0,0
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071,0


## Concat histone columns into single column

In [31]:
gene_w_histone = gene_w_histone.with_columns(
    pl.struct(['H3K4me3', 'H3K4me1','H3K36me3',  'H3K9me3', 'H3K27me3']).map_elements(
        lambda x: [x['H3K4me1'], x['H3K4me3'], x['H3K9me3'], x['H3K27me3'], x['H3K36me3']],
        return_dtype = pl.List(pl.List(pl.Float64))
    )
    .alias('all_histone')
)

In [32]:
gene_w_histone

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K36me3,H3K36me3_wc,H3K36me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E066,label,all_histone
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,i64,list[list[f64]]
"""ENSG00000000003""","[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",9,100,"[3.39243, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,73.205,3,"[[0.0, 0.0, … 0.0], [0.0, 4.19024, … 0.0], … [3.39243, 0.0, … 0.0]]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,0.191,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",52,100,"[10.55422, 10.55422, … 0.0]",25,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,52.609,3,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [10.55422, 10.55422, … 0.0]]"
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,4.733,2,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.0104, 3.0104, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,0.942,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",5,100,0.0,0,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071,0,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"


## Save into parquet file

In [33]:
# Save to parquet
gene_w_histone.write_parquet(os.path.join(DATASET_PATH, 'E066_w_histone_qlabel_pl.parquet'))

## Read from parquet

In [34]:
E066_w_histone_pl = pl.read_parquet(os.path.join(DATASET_PATH, 'E066_w_histone_qlabel_pl.parquet'))

In [35]:
E066_w_histone_pl

gene_id,H3K4me3,H3K4me3_wc,H3K4me3_len,H3K4me1,H3K4me1_wc,H3K4me1_len,H3K36me3,H3K36me3_wc,H3K36me3_len,H3K9me3,H3K9me3_wc,H3K9me3_len,H3K27me3,H3K27me3_wc,H3K27me3_len,E066,label,all_histone
str,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,list[f64],u32,u32,f64,i64,list[list[f64]]
"""ENSG00000000003""","[0.0, 4.19024, … 0.0]",19,100,"[0.0, 0.0, … 0.0]",9,100,"[3.39243, 0.0, … 0.0]",1,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,73.205,3,"[[0.0, 0.0, … 0.0], [0.0, 4.19024, … 0.0], … [3.39243, 0.0, … 0.0]]"
"""ENSG00000000005""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,0.191,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000000419""","[0.0, 0.0, … 0.0]",36,100,"[0.0, 0.0, … 0.0]",52,100,"[10.55422, 10.55422, … 0.0]",25,100,"[0.0, 0.0, … 0.0]",2,100,"[0.0, 0.0, … 0.0]",0,100,52.609,3,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [10.55422, 10.55422, … 0.0]]"
"""ENSG00000000457""","[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",35,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,4.733,2,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000000460""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[3.0104, 3.0104, … 0.0]",5,100,"[0.0, 0.0, … 0.0]",0,100,0.942,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""ENSG00000259658""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.212,1,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000259664""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",5,100,0.0,0,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"
"""ENSG00000259680""","[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,"[0.0, 0.0, … 0.0]",0,100,0.071,0,"[[0.0, 0.0, … 0.0], [0.0, 0.0, … 0.0], … [0.0, 0.0, … 0.0]]"


# [EXPERIMENT] Polars 'join' inconsistent results

Test with small dataset

## Using manual step

In [ ]:
# gene_sample = E066_pl.filter(pl.col('gene_id').is_in(['ENSG00000259133', 'ENSG00000016082']))
gene_sample = E066_pl.limit(200)

In [ ]:
gene_sample

In [ ]:
gene_sample = gene_sample.with_columns([
        pl.int_ranges(pl.col('start'), pl.col('end'), 100).alias('window_starts')
    ]).explode('window_starts')

In [ ]:
gene_sample = gene_sample.with_columns([
        (pl.col('window_starts') + 100).alias('window_ends')
    ])

In [ ]:
gene_sample

In [ ]:
# Join genes with histone data
joined = gene_sample.join(
    H3K4me1_pl,
    left_on='gene_id',
    right_on='gene_id',
    how='left'
)

In [ ]:
joined

In [ ]:
# Filter and calculate average signal value
result = joined.filter(
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') >= pl.col('window_starts')) & 
    (pl.col('chromStart') <= pl.col('window_ends')) & 
    (pl.col('chromEnd') >= pl.col('window_ends')) |
    (pl.col('chromStart') <= pl.col('window_starts')) & 
    (pl.col('chromEnd') >= pl.col('window_starts')) & 
    (pl.col('chromEnd') <= pl.col('window_ends'))
).group_by(['gene_id', 'window_starts'], maintain_order=True).agg([
    pl.col('signalValue').mean().alias('H3K4me1')
]).sort(['gene_id', 'window_starts'])

In [ ]:
result

In [ ]:
# Find the gene without histone match
gene_wo_histone = gene_sample.filter(~pl.col('window_starts').is_in(result['window_starts']))

In [ ]:
gene_wo_histone

In [ ]:
# Restructured the dataframe so it can be merged with results
gene_wo_histone = gene_wo_histone.with_columns(
    chromosome_name_right = pl.lit(None).cast(pl.String),
    start_right = pl.lit(0).cast(pl.Int64),
    end_right =  pl.lit(0).cast(pl.Int64),
    E066_right = pl.lit(0.0).cast(pl.Float64),
    strand_right = pl.lit(0).cast(pl.Int64),
    label_right = pl.lit(0).cast(pl.Int64),
    external_gene_name_right = pl.lit(None).cast(pl.String),
    start_position_right = pl.lit(0).cast(pl.Int64),
    end_position_right = pl.lit(0).cast(pl.Int64),
    tss_right = pl.lit(0).cast(pl.Int64),
    chrom = pl.lit(None).cast(pl.String),
    chromStart = pl.lit(0).cast(pl.Int64),
    chromEnd = pl.lit(0).cast(pl.Int64),
    name = pl.lit(None).cast(pl.String),
    score = pl.lit(0).cast(pl.Int64),
    strand_peak = pl.lit(None).cast(pl.String),
    signalValue = pl.lit(0.0).cast(pl.Float64),
    pValue = pl.lit(0.0).cast(pl.Float64),
    qValue = pl.lit(0.0).cast(pl.Float64),
    peak = pl.lit(0).cast(pl.Int64),
    startBucket = pl.lit(0.0).cast(pl.Float64),
    endBucket = pl.lit(0.0).cast(pl.Float64)
)

In [ ]:
gene_wo_histone

In [ ]:
# Grouping so it can be merged with result dataframe
gene_wo_histone = gene_wo_histone.group_by(['gene_id', 'window_starts']).agg([
    pl.col('signalValue').mean().alias('H3K4me1')
]).sort(['gene_id', 'window_starts'])

In [ ]:
gene_wo_histone

In [ ]:
# Merging with the result dataframe
result.extend(gene_wo_histone)

# Sorting by window_start
result = result.sort("window_starts")

In [ ]:
result

In [ ]:
# Final group by
result = result.group_by(['gene_id'], maintain_order=True).agg(
    pl.col('H3K4me1')
).sort('gene_id')

In [ ]:
result

In [ ]:
result = result.with_columns(
    pl.col('H3K4me1')
    .list.eval(pl.element().is_not_null() & (pl.element() > 0))
    .list.sum()
    .alias("H3K4me1_wc")
)

In [ ]:
result = result.with_columns(
    pl.col('H3K4me1').list.len().alias("H3K4me1_len")
)

In [ ]:
result

In [ ]:
result.filter(pl.col('H3K4me1_len') < 100)

## Using function

### Manual selection

In [ ]:
# gene_sample2 = E066_pl.filter(pl.col('gene_id').is_in(['ENSG00000259133', 'ENSG00000016082']))
gene_sample2 = E066_pl.limit(300)
# gene_sample2 = E066_pl.gather_every(1000, offset=0)

In [ ]:
gene_sample2

In [ ]:
result2 = get_histone_features(gene_sample2, H3K4me1_pl, 'H3K4me1')

In [ ]:
result2.schema

In [ ]:
result2

In [ ]:
len(result2.filter(pl.col('H3K4me1_len') < 100))

In [ ]:
result2.schema

### Iter Slice

#### H3K4me1

In [ ]:
histone_name = 'H3K4me1'
H3K4me1_schema = pl.Schema({
    'gene_id': pl.String,
    histone_name: pl.List(pl.Float64),
    f'{histone_name}_wc': pl.UInt32,
    f'{histone_name}_len': pl.UInt32
})

In [ ]:
gene_w_H3K4me1 = pl.DataFrame(schema=H3K4me1_schema)

In [ ]:
gene_w_H3K4me1

In [ ]:
error_rows = 0

gene_w_H3K4me1 = pl.DataFrame(schema=H3K4me1_schema)

for i, chunk in enumerate(E066_pl.iter_slices(n_rows=20_000)):
    # print(f"Chunk {i+1}: , Chunk size: {len(chunk)}")
    result = get_histone_features(chunk, H3K4me1_pl, 'H3K4me1')
    non_100 = len(result.filter(pl.col('H3K4me1_len') < 100))
    if non_100 > 0:
        print(f"[ERROR] Chunk {i+1}: {non_100}")
        error_rows += non_100
    else:
        print(f"Chunk {i+1}: {non_100}")
    # print(result.head(5))
    # print("---")

    gene_w_H3K4me1.extend(result)

print("---")
print(f"Error rows: {error_rows}")

In [ ]:
gene_w_H3K4me1.filter(pl.col('H3K4me1_len') < 100)